##Citibike Preprocessing

In [33]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
df = pd.read_csv('../data/citibike_weather.csv')

C1: Fix the Dtypes
Fix every dtype problem you found in E1: dates should be datetime, numbers should be numeric.
 Tip: pd.to_datetime() and pd.to_numeric() are your friends. If a numeric conversion fails, look at WHY before reaching for errors='coerce' — know what you're coercing.



In [34]:
#Fix the Dtypes
df['ride_date'] = pd.to_datetime(df['ride_date'])
df['temp_f'] = pd.to_numeric(df['temp_f'], errors='coerce')
df['max_temp_f'] = pd.to_numeric(df['max_temp_f'], errors='coerce')
df['min_temp_f'] = pd.to_numeric(df['min_temp_f'], errors='coerce')
df['wind_speed_knots'] = pd.to_numeric(df['wind_speed_knots'], errors='coerce')
df['precip_in'] = pd.to_numeric(df['precip_in'], errors='coerce')
print("Dtypes after conversion:")
print(df.dtypes)

Dtypes after conversion:
ride_date           datetime64[us]
num_rides                    int64
avg_duration_min           float64
temp_f                     float64
max_temp_f                 float64
min_temp_f                 float64
wind_speed_knots           float64
precip_in                  float64
day_of_week                    str
month                        int64
dtype: object


C2: Handle the Coded Missing Values
Deal with whatever your E2 sentinel hunt turned up. First convert any coded values to proper NaN, then decide: drop the row(s), or impute? Justify your choice in a markdown cell — there is more than one defensible answer, but “I didn't notice” is not one of them.
 Tip: Think about how many rows are affected and what imputation would be reasonable for that variable (e.g., a nearby day's value, a median, or zero — which makes sense for THIS variable?).



In [35]:
#Check for missing values
missing_values = df.isnull().sum()
print("Missing values per column:")
print(missing_values) 


Missing values per column:
ride_date           0
num_rides           0
avg_duration_min    0
temp_f              0
max_temp_f          0
min_temp_f          0
wind_speed_knots    0
precip_in           0
day_of_week         0
month               0
dtype: int64


In [36]:
# C2: replace NOAA sentinel value with NaN, then impute
df['precip_in'] = df['precip_in'].replace(99.99, np.nan)
print("Missing values per column after sentinel replacement:")
print(df.isna().sum())

Missing values per column after sentinel replacement:
ride_date           0
num_rides           0
avg_duration_min    0
temp_f              0
max_temp_f          0
min_temp_f          0
wind_speed_knots    0
precip_in           1
day_of_week         0
month               0
dtype: int64


In [37]:
median_precip = df['precip_in'].median()
df['precip_in'] = df['precip_in'].fillna(median_precip)
print(f"Imputed missing precip_in with median: {median_precip}")

Imputed missing precip_in with median: 0.0


Only precip_in had a coded sentinel (99.99), affecting 1 of 1,610 rows. Because rainfall is heavily right-skewed (most days have 0 inches), the median is a safer fill than the mean — imputed value was 0.0 inches."

In [38]:
missing_values = df.isna().sum()
print("Missing values per column:")
print(missing_values)


Missing values per column:
ride_date           0
num_rides           0
avg_duration_min    0
temp_f              0
max_temp_f          0
min_temp_f          0
wind_speed_knots    0
precip_in           0
day_of_week         0
month               0
dtype: int64


C3: Encode Day of Week
Your model can't multiply 'Tuesday' by a coefficient. One-hot encode day_of_week into indicator columns.
 Tip: pd.get_dummies(). Look up what drop_first=True does and decide whether to use it — either choice is fine if you can say why.



In [43]:
# one-hot encode day_of_week
df= pd.get_dummies(df, columns=['day_of_week'], drop_first=True)
print("Dummies created for day_of_week (Friday is the baseline).")




Dummies created for day_of_week (Friday is the baseline).


C4: Build a Trend Feature
Give your model a way to know about the system growth you found in E4. Create either a year column or a days-since-launch index (or both, and pick one for modeling).
 Tip: If ride_date is a proper datetime, .dt.year is one option; subtracting the first date and taking .dt.days is another.



In [40]:
#Build trend features
df['days_since_start'] = (df['ride_date'] - df['ride_date'].min()).dt.days
print("Trend feature created: days_since_start, range", df['days_since_start'].min(), "to", df['days_since_start'].max())


Trend feature created: days_since_start, range 0 to 1795


C5 (Stretch): Engineer Smarter Features
Optional, for those who want to push the model further. Ideas: a squared temperature term (revisit what you saw in E3 on the hottest days); an is_weekend flag; a rained-at-all binary flag; a US-holidays flag. Each one you add, justify with one sentence tying it to something you observed in EDA.
 Tip: A squared feature is how a LINEAR model captures a CURVED relationship — the model is still linear in its coefficients.



C6: Save the Clean Dataset
Save your finished dataframe to data/citibike_weather_daily_clean.csv. This file is what model.ipynb loads — the raw CSV's job is done.



In [44]:
df.to_csv('../data/citibike_weather_preprocessed.csv', index=False)
print("Saved clean dataset:", df.shape)
print(df.shape)
print(df.columns.tolist())

Saved clean dataset: (1610, 16)
(1610, 16)
['ride_date', 'num_rides', 'avg_duration_min', 'temp_f', 'max_temp_f', 'min_temp_f', 'wind_speed_knots', 'precip_in', 'month', 'days_since_start', 'day_of_week_Monday', 'day_of_week_Saturday', 'day_of_week_Sunday', 'day_of_week_Thursday', 'day_of_week_Tuesday', 'day_of_week_Wednesday']
